In [ ]:

# ============================================================
# FOODVISION AI v1.3
# Three AI Models | Google Colab | Single Cell
# ============================================================

# 1. INSTALL
!pip -q install ultralytics gradio "transformers==4.57.3" accelerate huggingface_hub

# 2. IMPORT
import time
import requests
import torch
import gradio as gr

from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    AutoImageProcessor,
    AutoModelForImageClassification
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DEVICE_NAME = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

print("FoodVision AI v1.3")
print("Device:", DEVICE_NAME)


# 3. MODEL CONFIGURATION

YOLO_FILENAME = "yolov8n-cls.pt"

YOLO_URLS = [
    "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n-cls.pt",
    "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n-cls.pt",
    "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n-cls.pt"
]

MODEL_IDS = {
    "YOLOv8n-cls": YOLO_FILENAME,
    "Salesforce BLIP": "Salesforce/blip-image-captioning-base",
    "nateraw/food": "nateraw/food"
}

MODELS = {name: None for name in MODEL_IDS}


# 4. DOWNLOAD YOLO CHECKPOINT WITH FALLBACK

def download_yolo():

    local_file = Path("/content") / YOLO_FILENAME

    if local_file.exists() and local_file.stat().st_size > 1_000_000:
        print("Using cached YOLOv8 weights")
        return str(local_file)

    for url in YOLO_URLS:

        print("Trying:", url)

        for attempt in range(2):

            temp_file = local_file.with_suffix(".pt.part")

            try:

                response = requests.get(
                    url,
                    stream=True,
                    timeout=(30, 180)
                )

                response.raise_for_status()

                total = 0

                with open(temp_file, "wb") as f:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)
                            total += len(chunk)

                if total < 1_000_000:
                    raise RuntimeError("Invalid checkpoint size")

                temp_file.replace(local_file)

                print(
                    f"YOLOv8 downloaded: "
                    f"{total / 1024 / 1024:.2f} MB"
                )

                return str(local_file)

            except Exception as e:

                temp_file.unlink(missing_ok=True)

                print(
                    f"Download attempt {attempt + 1} failed: {e}"
                )

    raise RuntimeError(
        "Unable to download YOLOv8 checkpoint. "
        "Please try again later."
    )


# 5. LOAD MODELS

def load_model(name):

    if MODELS[name] is not None:
        return MODELS[name]

    print("Loading:", name)

    if name == "YOLOv8n-cls":

        weights = download_yolo()
        model = YOLO(weights)

    elif name == "Salesforce BLIP":

        processor = BlipProcessor.from_pretrained(
            MODEL_IDS[name]
        )

        model = BlipForConditionalGeneration.from_pretrained(
            MODEL_IDS[name]
        )

        model.to(DEVICE)
        model.eval()

        model = (processor, model)

    elif name == "nateraw/food":

        processor = AutoImageProcessor.from_pretrained(
            MODEL_IDS[name]
        )

        model = AutoModelForImageClassification.from_pretrained(
            MODEL_IDS[name]
        )

        model.to(DEVICE)
        model.eval()

        model = (processor, model)

    MODELS[name] = model

    print(name, "loaded successfully!")

    return model


# 6. YOLOv8 CLASSIFICATION

def predict_yolo(image):

    model = load_model("YOLOv8n-cls")

    result = model.predict(
        source=image,
        imgsz=224,
        verbose=False
    )[0]

    probs = result.probs

    indices = torch.topk(
        probs.data,
        k=5
    ).indices.tolist()

    output = "### YOLOv8 Top 5 Predictions\n\n"

    for idx in indices:

        label = result.names[int(idx)]
        score = float(probs.data[int(idx)])

        output += f"- **{label}** — {score:.2%}\n"

    return output


# 7. SALESFORCE BLIP

def predict_blip(image):

    processor, model = load_model("Salesforce BLIP")

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=60
        )

    caption = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    return (
        "### BLIP Image Description\n\n"
        + caption
    )


# 8. NATERAW FOOD CLASSIFICATION

def predict_nateraw(image):

    processor, model = load_model("nateraw/food")

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        outputs = model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )[0]

        top = torch.topk(
            probabilities,
            k=5
        )

    output = "### nateraw/food Top 5 Predictions\n\n"

    for idx, score in zip(
        top.indices.tolist(),
        top.values.tolist()
    ):

        label = model.config.id2label.get(
            idx, str(idx)
        )

        output += f"- **{label}** — {score:.2%}\n"

    return output


# 9. COMMON EXECUTION ENGINE

def run_model(name, image):

    if image is None:
        raise ValueError("Please upload a food image.")

    image = image.convert("RGB")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    if name == "YOLOv8n-cls":

        result = predict_yolo(image)

    elif name == "Salesforce BLIP":

        result = predict_blip(image)

    elif name == "nateraw/food":

        result = predict_nateraw(image)

    else:

        raise ValueError("Unknown model")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    return result, elapsed


# 10. INDIVIDUAL MODEL TEST

def run_selected(name, image):

    try:

        result, elapsed = run_model(name, image)

        return (
            result,
            f"{elapsed:.2f} seconds"
        )

    except Exception as e:

        print(name, "ERROR:", repr(e))

        return (
            f"### Error: {name}\n\n"
            f"```text\n{type(e).__name__}: {e}\n```",
            "Failed"
        )


# 11. BENCHMARK ALL THREE

def run_benchmark(image):

    if image is None:
        return "Please upload an image.", []

    report = (
        "# FoodVision Benchmark\n\n"
        f"**Device:** {DEVICE_NAME}\n\n"
    )

    table = []

    for name in MODEL_IDS:

        try:

            result, elapsed = run_model(
                name,
                image
            )

            report += (
                f"## {name}\n\n"
                f"**Time:** {elapsed:.2f} seconds\n\n"
                f"{result}\n\n"
                "---\n\n"
            )

            table.append([
                name,
                f"{elapsed:.2f} sec",
                "Success"
            ])

        except Exception as e:

            error = f"{type(e).__name__}: {e}"

            print(name, "BENCHMARK ERROR:", error)

            report += (
                f"## {name} — FAILED\n\n"
                f"```text\n{error}\n```\n\n"
                "---\n\n"
            )

            table.append([
                name,
                "-",
                error[:150]
            ])

    return report, table


# 12. GRADIO APPLICATION

with gr.Blocks() as demo:

    gr.Markdown("""
    # 🍔 FoodVision AI v1.3

    ### Three AI Models. One Food Image.

    Experiment with three different pretrained AI models.
    """)

    image_input = gr.Image(
        type="pil",
        label="Upload Food Image"
    )

    with gr.Tab("Individual Model"):

        model_choice = gr.Dropdown(
            choices=list(MODEL_IDS.keys()),
            value="YOLOv8n-cls",
            label="Select AI Model"
        )

        run_button = gr.Button(
            "Run Model",
            variant="primary"
        )

        individual_output = gr.Markdown()

        execution_time = gr.Textbox(
            label="Execution Time",
            interactive=False
        )

        run_button.click(
            run_selected,
            inputs=[
                model_choice,
                image_input
            ],
            outputs=[
                individual_output,
                execution_time
            ]
        )

    with gr.Tab("Benchmark"):

        benchmark_button = gr.Button(
            "Benchmark All Three Models",
            variant="primary"
        )

        benchmark_output = gr.Markdown()

        benchmark_table = gr.Dataframe(
            headers=[
                "Model",
                "Execution Time",
                "Status"
            ],
            interactive=False
        )

        benchmark_button.click(
            run_benchmark,
            inputs=[image_input],
            outputs=[
                benchmark_output,
                benchmark_table
            ]
        )

    gr.Markdown("""
    ---
    **FoodVision AI v1.3 | Phase 1**

    Ultralytics · Hugging Face · Gradio
    """)


# 13. LAUNCH

print("Launching FoodVision AI...")

demo.launch(
    share=True,
    debug=True,
    show_error=True,
    theme=gr.themes.Soft()
)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


FoodVision AI v1.3
Device: CPU
Launching FoodVision AI...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1c505832544c699db4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading: YOLOv8n-cls
Trying: https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n-cls.pt
Download attempt 1 failed: 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n-cls.pt
Download attempt 2 failed: 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n-cls.pt
Trying: https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n-cls.pt
Download attempt 1 failed: 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n-cls.pt
Download attempt 2 failed: 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n-cls.pt
Trying: https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n-cls.pt
YOLOv8 downloaded: 5.31 MB
YOLOv8n-cls loaded successfully!


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading: Salesforce BLIP


... (12 outputs left out)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/344M [00:00<?, ?B/s]

nateraw/food loaded successfully!


model.safetensors:   0%|          | 0.00/344M [00:00<?, ?B/s]